In [2]:
%pip install -q pandas scikit-learn streamlit joblib requests

import pandas
import sklearn
import streamlit
import joblib
import requests

print(pandas.__version__)
print(sklearn.__version__)
print(streamlit.__version__)
print(joblib.__version__)
print(requests.__version__)

Note: you may need to restart the kernel to use updated packages.
2.3.2
1.7.2
1.52.1
1.5.2
2.32.5


In [3]:
from pathlib import Path
import requests
import pandas as pd

Path("data").mkdir(exist_ok=True)

file_path = Path("data/results.csv")
if not file_path.exists():
    url = "https://raw.githubusercontent.com/IBM-SkillsBuild-AI-Builders-Challenge/hands-on-labs/main/02_football_lab_june/04_data/results.csv"
    response = requests.get(url)
    file_path.write_bytes(response.content)

matches = pd.read_csv("data/results.csv", parse_dates=["date"])

print(matches.shape)
print(matches["date"].min())
print(matches["date"].max())
display(matches.head(3))

(49353, 9)
1872-11-30 00:00:00
2026-06-27 00:00:00


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False


In [4]:
print("Top 10 most frequent tournaments:")
print(matches["tournament"].value_counts().head(10))
print()

print("Top 15 teams by total matches played:")
home_counts = matches["home_team"].value_counts()
away_counts = matches["away_team"].value_counts()
total_counts = home_counts.add(away_counts, fill_value=0).astype(int)
print(total_counts.sort_values(ascending=False).head(15))
print()

print("Number of matches per decade:")
matches_decade = matches.copy()
matches_decade["decade"] = (matches_decade["date"].dt.year // 10 * 10).astype(str) + "s"
decade_counts = matches_decade["decade"].value_counts().sort_index()
print(decade_counts)

Top 10 most frequent tournaments:
tournament
Friendly                                18279
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
Name: count, dtype: int64

Top 15 teams by total matches played:
Sweden         1102
England        1091
Argentina      1069
Brazil         1061
Germany        1033
South Korea    1009
Mexico         1005
Hungary        1004
Uruguay         973
France          936
Poland          891
Italy           891
Switzerland     886
Netherlands     880
Norway          873
Name: count, dtype: int64

Number of matches per decade:
decade
1870s      13
1880s      55
1890s      59
1900s     137
1910s     

In [5]:
def winrate(hist):
    if len(hist) == 0:
        return 0.5
    return sum(w for _, _, w in hist) / len(hist)

def goal_avg(hist):
    if len(hist) == 0:
        return 1.0
    return sum(gf for gf, _, _ in hist) / len(hist)

def recent_form(hist):
    if len(hist) < 10:
        return 0.5
    recent = hist[-10:]
    return sum(w for _, _, w in recent) / 10

filtered = matches[matches["date"] >= "1990-01-01"].sort_values("date").reset_index(drop=True)

team_history = {}
major_tournaments = {"Soccer World Cup", "Soccer World Cup qualification", "UEFA Euro", "UEFA Euro qualification", "Copa América", "African Cup of Nations"}

rows = []
for _, row in filtered.iterrows():
    home = row["home_team"]
    away = row["away_team"]
    
    if home not in team_history:
        team_history[home] = []
    if away not in team_history:
        team_history[away] = []
    
    home_hist = team_history[home]
    away_hist = team_history[away]
    
    team_a_winrate = winrate(home_hist)
    team_b_winrate = winrate(away_hist)
    team_a_goal_avg = goal_avg(home_hist)
    team_b_goal_avg = goal_avg(away_hist)
    team_a_recent_form = recent_form(home_hist)
    team_b_recent_form = recent_form(away_hist)
    is_neutral = int(row["neutral"])
    is_major_tournament = 1 if row["tournament"] in major_tournaments else 0
    
    home_score = row["home_score"]
    away_score = row["away_score"]
    if home_score > away_score:
        outcome = 0
    elif home_score == away_score:
        outcome = 1
    else:
        outcome = 2
    
    rows.append({
        "date": row["date"],
        "home_team": home,
        "away_team": away,
        "team_a_winrate": team_a_winrate,
        "team_b_winrate": team_b_winrate,
        "team_a_goal_avg": team_a_goal_avg,
        "team_b_goal_avg": team_b_goal_avg,
        "team_a_recent_form": team_a_recent_form,
        "team_b_recent_form": team_b_recent_form,
        "is_neutral": is_neutral,
        "is_major_tournament": is_major_tournament,
        "outcome": outcome
    })
    
    home_won = 1 if home_score > away_score else 0
    away_won = 1 if away_score > home_score else 0
    
    team_history[home].append((home_score, away_score, home_won))
    team_history[away].append((away_score, home_score, away_won))

features_df = pd.DataFrame(rows)

print(features_df.shape)
features_df.head(3)

(32236, 12)


,date,home_team,away_team,team_a_winrate,team_b_winrate,team_a_goal_avg,team_b_goal_avg,team_a_recent_form,team_b_recent_form,is_neutral,is_major_tournament,outcome
0,1990-01-12,Algeria,Mali,0.5,0.5,1.0,1.0,0.5,0.5,1,0,0
1,1990-01-14,Algeria,Cameroon,1.0,0.5,5.0,1.0,0.5,0.5,1,0,0
2,1990-01-17,Greece,Belgium,0.5,0.5,1.0,1.0,0.5,0.5,0,0,0


In [6]:
feature_cols = ["team_a_winrate", "team_b_winrate", "team_a_goal_avg", "team_b_goal_avg", "team_a_recent_form", "team_b_recent_form", "is_neutral", "is_major_tournament"]

train_mask = features_df["date"] < "2018-01-01"
test_mask = features_df["date"] >= "2018-01-01"

X_train = features_df.loc[train_mask, feature_cols]
X_test = features_df.loc[test_mask, feature_cols]
y_train = features_df.loc[train_mask, "outcome"]
y_test = features_df.loc[test_mask, "outcome"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print()
print("Class distribution in y_train:")
print(y_train.value_counts(normalize=True))

X_train shape: (24179, 8)
X_test shape: (8057, 8)
y_train shape: (24179,)
y_test shape: (8057,)

Class distribution in y_train:
outcome
0    0.487117
2    0.276273
1    0.236610
Name: proportion, dtype: float64


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_acc * 100:.2f}%")

most_frequent_class = y_train.value_counts().idxmax()
baseline_acc = (y_test == most_frequent_class).mean()
print(f"Baseline accuracy (always predict most frequent class): {baseline_acc * 100:.2f}%")
print()

cm = confusion_matrix(y_test, y_pred)
labels = ["Home win", "Draw", "Away win"]
print("Confusion Matrix (rows=actual, cols=predicted):")
print("\t\t" + "\t".join(labels))
for i, label in enumerate(labels):
    print(f"{label}\t\t" + "\t".join(str(cm[i, j]) for j in range(len(labels))))
print()

print("Feature importances (sorted descending):")
importances = sorted(zip(feature_cols, model.feature_importances_), key=lambda x: x[1], reverse=True)
for feature, importance in importances:
    print(f"{feature}: {importance:.4f}")

Test accuracy: 56.13%
Baseline accuracy (always predict most frequent class): 47.29%

Confusion Matrix (rows=actual, cols=predicted):
		Home win	Draw	Away win
Home win		3404	35	371
Draw		1389	33	418
Away win		1270	52	1085

Feature importances (sorted descending):
team_b_winrate: 0.2307
team_a_winrate: 0.2113
team_b_goal_avg: 0.1868
team_a_goal_avg: 0.1817
team_a_recent_form: 0.0745
team_b_recent_form: 0.0743
is_neutral: 0.0268
is_major_tournament: 0.0138


In [8]:
from pathlib import Path
import joblib

# Create models directory
Path("models").mkdir(exist_ok=True)

# Build set of Soccer World Cup qualification teams
wc_qual_mask = matches["tournament"] == "Soccer World Cup qualification"
soccer_teams = set(matches.loc[wc_qual_mask, "home_team"]) | set(matches.loc[wc_qual_mask, "away_team"])

# Calculate team statistics
team_stats = {}

for team in soccer_teams:
    # Get all matches for this team
    home_matches = matches[matches["home_team"] == team]
    away_matches = matches[matches["away_team"] == team]
    
    total_matches = len(home_matches) + len(away_matches)
    
    # Skip teams with fewer than 30 matches
    if total_matches < 30:
        continue
    
    # Calculate wins
    home_wins = (home_matches["home_score"] > home_matches["away_score"]).sum()
    away_wins = (away_matches["away_score"] > away_matches["home_score"]).sum()
    total_wins = home_wins + away_wins
    winrate = total_wins / total_matches
    
    # Calculate goal average
    home_goals = home_matches["home_score"].sum()
    away_goals = away_matches["away_score"].sum()
    total_goals = home_goals + away_goals
    goal_avg = total_goals / total_matches
    
    # Calculate recent form (last 10 matches by date)
    all_team_matches = pd.concat([
        home_matches.assign(team_score=home_matches["home_score"], opp_score=home_matches["away_score"]),
        away_matches.assign(team_score=away_matches["away_score"], opp_score=away_matches["home_score"])
    ]).sort_values("date")
    
    if len(all_team_matches) < 10:
        recent_form = 0.5
    else:
        last_10 = all_team_matches.tail(10)
        recent_wins = (last_10["team_score"] > last_10["opp_score"]).sum()
        recent_form = recent_wins / 10
    
    team_stats[team] = {
        "winrate": float(winrate),
        "goal_avg": float(goal_avg),
        "recent_form": float(recent_form),
        "matches_played": int(total_matches)
    }

# Save model and team data
joblib.dump(model, "models/match_predictor.pkl")
joblib.dump({"team_stats": team_stats, "feature_cols": feature_cols}, "models/team_data.pkl")

# Print statistics
print(f"Number of teams stored: {len(team_stats)}")
print()
print("Top 5 teams by winrate (≥100 matches):")
top_teams = [(team, stats["winrate"]) for team, stats in team_stats.items() if stats["matches_played"] >= 100]
top_teams.sort(key=lambda x: x[1], reverse=True)
for i, (team, wr) in enumerate(top_teams[:5], 1):
    matches_played = team_stats[team]["matches_played"]
    print(f"{i}. {team}: {wr:.4f} ({matches_played} matches)")

Number of teams stored: 0

Top 5 teams by winrate (≥100 matches):


In [9]:
def predict_match(team_a, team_b, is_neutral=True, is_major_tournament=True):
    """
    Predict match outcome probabilities between two teams.
    
    Args:
        team_a: Name of first team (home team position)
        team_b: Name of second team (away team position)
        is_neutral: Whether match is at neutral venue (default: True)
        is_major_tournament: Whether match is in major tournament (default: True)
    
    Returns:
        Dictionary with win/draw probabilities for both teams
    """
    # Validate teams exist in team_stats
    if team_a not in team_stats:
        raise ValueError(f"Team '{team_a}' not found in team statistics. Please use a team that has played in Soccer World Cup qualification.")
    if team_b not in team_stats:
        raise ValueError(f"Team '{team_b}' not found in team statistics. Please use a team that has played in Soccer World Cup qualification.")
    
    # Get team statistics
    stats_a = team_stats[team_a]
    stats_b = team_stats[team_b]
    
    # Build feature row
    features = pd.DataFrame([{
        "team_a_winrate": stats_a["winrate"],
        "team_b_winrate": stats_b["winrate"],
        "team_a_goal_avg": stats_a["goal_avg"],
        "team_b_goal_avg": stats_b["goal_avg"],
        "team_a_recent_form": stats_a["recent_form"],
        "team_b_recent_form": stats_b["recent_form"],
        "is_neutral": int(is_neutral),
        "is_major_tournament": int(is_major_tournament)
    }])
    
    # Reindex to match training column order
    features = features[feature_cols]
    
    # Get predictions
    proba = model.predict_proba(features)
    
    # Return probabilities (0=team_a win, 1=draw, 2=team_b win)
    return {
        "team_a_win_prob": float(proba[0][0]),
        "draw_prob": float(proba[0][1]),
        "team_b_win_prob": float(proba[0][2])
    }

# Test the function
print("Prediction for Brazil vs Argentina:")
result1 = predict_match("Brazil", "Argentina")
print(f"  Brazil win: {result1['team_a_win_prob']:.2%}")
print(f"  Draw: {result1['draw_prob']:.2%}")
print(f"  Argentina win: {result1['team_b_win_prob']:.2%}")
print()

print("Prediction for Germany vs Brazil:")
result2 = predict_match("Germany", "Brazil")
print(f"  Germany win: {result2['team_a_win_prob']:.2%}")
print(f"  Draw: {result2['draw_prob']:.2%}")
print(f"  Brazil win: {result2['team_b_win_prob']:.2%}")

Prediction for Brazil vs Argentina:


ValueError: Team 'Brazil' not found in team statistics. Please use a team that has played in Soccer World Cup qualification.